# nf-core/demo

A simple demonstration pipeline for learning Nextflow and nf-core.

## Overview

The `nf-core/demo` pipeline is designed as a learning tool. It demonstrates:
- Basic Nextflow workflow structure
- Input/output handling
- Process definitions
- Container usage
- nf-core patterns

**Execution time:** ~2-5 minutes for test profile

**Requirements:**
- Nextflow installed
- Docker or Singularity (for containers)
- OR conda environment

## Installation

In [3]:
# Install Nextflow via conda
# Run this once per environment
!conda install -c bioconda nextflow -y

Channels:
 - bioconda
 - conda-forge
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 26.3.2
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /opt/conda

  added / updated specs:
    - nextflow


The following packages will be UPDATED:

  nextflow                               24.04.4-hdfd78af_0 --> 26.04.4-h2a3209d_0 




Preparing transaction: done
Verifying transaction: done
Executing transaction: done


In [4]:
%%sh
# Verify installation
nextflow -version


      N E X T F L O W
      version 26.04.4 build 12445
      created 17-06-2026 16:30 UTC 
      cite doi:10.1038/nbt.3820
      http://nextflow.io



In [20]:
%%sh 
nextflow info 

  Version: 26.04.4 build 12445
  Created: 17-06-2026 16:30 UTC 
  System: Linux 6.12.88-119.157.amzn2023.x86_64
  Runtime: Groovy 4.0.31 on OpenJDK 64-Bit Server VM 21-internal-adhoc.conda.src
  Encoding: UTF-8 (UTF-8)



In [17]:
%%sh
set -e

cd ~
wget -q https://download.java.net/java/GA/jdk18/43f95e8614114aeaa8e8a5fcf20a682d/36/GPL/openjdk-18_linux-x64_bin.tar.gz
tar -xzf openjdk-18_linux-x64_bin.tar.gz
rm openjdk-18_linux-x64_bin.tar.gz 

In [18]:
%%sh
export JAVA_HOME=$HOME/jdk-18
export PATH=$JAVA_HOME/bin:$PATH
java -version   

openjdk version "18" 2022-03-22
OpenJDK Runtime Environment (build 18+36-2087)
OpenJDK 64-Bit Server VM (build 18+36-2087, mixed mode, sharing)


## Quick Start: Local Execution with Docker

This is the simplest way to run the demo pipeline on a SageMaker instance with Docker.

In [24]:
%%sh
# Create working directory
mkdir -p ~/demo_test_run
cd ~/demo_test_run

# Run demo pipeline with test profile
nextflow run nf-core/demo -r 1.2.0 -profile test,conda --outdir $PWD/results 
#nextflow run nf-core/demo -r 1.2.0\
#  -profile test \
#  -work-dir $PWD/work \
#  --outdir $PWD/results \
#  -with-report $PWD/report.html \
#  -with-timeline $PWD/timeline.html \
#  -with-trace $PWD/trace.txt


 N E X T F L O W   ~  version 26.04.4

WARN: This Nextflow version supports a new Multi-revision strategy for managing the SCM repositories, but 'nf-core/demo:1.2.0' is single-revision legacy strategy - Please consider to update the repository with the 'nextflow pull -migrate' command.
Launching `https://github.com/nf-core/demo` [spontaneous_turing] revision: 32893afef8 [1.2.0]

WARN: ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    There is a problem with your Conda configuration!
    You will need to set-up the conda-forge and bioconda channels correctly.
    Please refer to https://bioconda.github.io/
    The observed channel order is
    [conda-forge]
    but the following channel order is required:
    [conda-forge, bioconda]
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~"


------------------------------------------------------
                                        ,--./,-.
        ___     __   __   __   __

## Configuration Options

Choose the configuration that matches your execution environment.

### Option 1: Local with Docker (Recommended)

In [ ]:
%%sh
mkdir -p ~/nextflow_config

cat > ~/nextflow_config/local_docker.config << 'EOF'
process {
    executor = 'local'
    cpus = 2
    memory = '8 GB'
}

docker {
    enabled = true
    runOptions = '-u $(id -u):$(id -g)'
}
EOF

echo "✅ Local Docker config created at ~/nextflow_config/local_docker.config"

### Option 2: AWS Batch Execution

For running on AWS Batch compute environment.

In [ ]:
# Set AWS credentials (temporary session credentials)
%env AWS_ACCESS_KEY_ID=enter_here_without_quotes
%env AWS_SECRET_ACCESS_KEY=enter_here_without_quotes
%env AWS_SESSION_TOKEN=enter_here_without_quotes

In [ ]:
%%sh
mkdir -p ~/nextflow_config

cat > ~/nextflow_config/aws_batch.config << 'EOF'
process {
    executor = 'awsbatch'
    queue = 'nextflow-base-v1'
}

aws {
    region = 'us-east-1'
    batch {
        cliPath = '/root/miniconda/bin/aws'
        jobRole = 'arn:aws:iam::335777049998:role/ecsTaskExecutionRole'
    }
}
EOF

echo "✅ AWS Batch config created at ~/nextflow_config/aws_batch.config"

## Run Pipeline

### Local Execution with Docker

In [ ]:
%%sh
# Create working directory
mkdir -p ~/demo_test_run
cd ~/demo_test_run

# Run demo pipeline
nextflow run nf-core/demo \
  -profile test,docker \
  -c ~/nextflow_config/local_docker.config \
  -work-dir $PWD/work \
  --outdir $PWD/results \
  -with-report $PWD/report.html \
  -with-timeline $PWD/timeline.html \
  -with-trace $PWD/trace.txt

### AWS Batch Execution

In [ ]:
%%sh
# Run demo pipeline on AWS Batch
# Results stored in S3

nextflow run nf-core/demo \
  -profile test \
  -c ~/nextflow_config/aws_batch.config \
  -work-dir s3://darp-dad-repos/RL/demo_workdir \
  --outdir s3://darp-dad-repos/RL/demo_output \
  -with-trace

### Resume Failed Run

If the pipeline fails or is interrupted, use `-resume` to continue from the last successful step.

In [ ]:
%%sh
cd ~/demo_test_run

# Add -resume flag to continue from last successful step
nextflow run nf-core/demo \
  -profile test,docker \
  -c ~/nextflow_config/local_docker.config \
  -work-dir $PWD/work \
  --outdir $PWD/results \
  -resume

## Check Results

In [ ]:
%%sh
# List output files
echo "=== Output Directory Structure ==="
ls -lR ~/demo_test_run/results

echo ""
echo "=== Execution Reports ==="
ls -lh ~/demo_test_run/*.html ~/demo_test_run/*.txt 2>/dev/null || echo "Reports generated in working directory"

In [ ]:
%%sh
# View execution trace summary
echo "=== Top 10 Tasks by Duration ==="
sort -t$'\t' -k4 -rn ~/demo_test_run/trace.txt | head -11 | cut -f1,2,4,5

## Understanding the Output

**Key output files:**
- `results/`: Main pipeline output directory
- `pipeline_info/`: Execution metadata and reports

**Execution reports:**
- `report.html`: Resource usage and task completion status
- `timeline.html`: Visual timeline of task execution
- `trace.txt`: Detailed metrics per task (CPU, memory, duration, status)

**Work directory:**
- `work/`: Contains intermediate files and task execution details
- Each task has a unique hash subdirectory
- Can be safely deleted after successful completion

## Exploring Pipeline Details

Learn more about the demo pipeline structure.

In [37]:
%%sh
# Show available parameters
nextflow run nf-core/demo -r 1.2.0 --help


 N E X T F L O W   ~  version 26.04.4

WARN: This Nextflow version supports a new Multi-revision strategy for managing the SCM repositories, but 'nf-core/demo:1.2.0' is single-revision legacy strategy - Please consider to update the repository with the 'nextflow pull -migrate' command.
Launching `https://github.com/nf-core/demo` [modest_darwin] revision: 32893afef8 [1.2.0]


------------------------------------------------------
                                        ,--./,-.
        ___     __   __   __   ___     /,-._.--~'
  |\ | |__  __ /  ` /  \ |__) |__         }  {
  | \| |       \__, \__/ |  \ |___     \`-._,-`-,
                                        `._,._,'
  nf-core/demo 1.2.0
------------------------------------------------------
Typical pipeline command:

  nextflow run nf-core/demo -profile <docker/singularity/.../institute> --input samplesheet.csv --outdir <OUTDIR>

help message of that parameter will be printed.  
or `--helpFull`.  

Input/output options
  --input   

In [38]:
%%sh
# List available profiles
echo "Common profiles:"
echo "  test       - Built-in test data"
echo "  docker     - Use Docker containers"
echo "  singularity - Use Singularity containers"
echo "  conda      - Use conda environments"
echo ""
echo "Profiles can be combined with commas: -profile test,docker"

Common profiles:
  test       - Built-in test data
  docker     - Use Docker containers
  singularity - Use Singularity containers
  conda      - Use conda environments

Profiles can be combined with commas: -profile test,docker


## Troubleshooting

### Common Issues

**1. Docker daemon not running**
```bash
# Check Docker status
docker ps

# Start Docker (requires sudo)
sudo systemctl start docker
```

**2. Pipeline not found**
- Ensure you have internet connectivity
- Nextflow will automatically download from GitHub
- Check nf-core/demo is available: https://nf-co.re/demo

**3. S3 Access Denied (AWS Batch)**
- Verify AWS credentials are set correctly
- Check IAM role has S3 permissions
- Ensure bucket name is correct

**4. Container pull failures**
- Check internet connectivity
- Verify Docker can pull images: `docker pull quay.io/nf-core/demo`

**5. Out of space errors**
```bash
# Clean up work directory
rm -rf ~/demo_test_run/work

# Clean Nextflow cache
nextflow clean -f
```

### View Logs

```bash
# Nextflow log
cat ~/demo_test_run/.nextflow.log

# Failed task details (replace <hash> with actual value from error message)
cd ~/demo_test_run/work/<hash>/
cat .command.log    # stdout
cat .command.err    # stderr
cat .command.sh     # executed script
```

## Next Steps

### Learn More

- [nf-core/demo documentation](https://nf-co.re/demo)
- [Nextflow patterns](https://nextflow-io.github.io/patterns/)
- [nf-core tutorials](https://nf-co.re/docs/usage/getting_started)
- [Nextflow training](https://training.nextflow.io/)

### Try Other Pipelines

Once you're comfortable with the demo, try these pipelines:
- `nf-core/rnaseq` - RNA sequencing analysis
- `nf-core/chipseq` - ChIP-seq analysis
- `nf-core/sarek` - Variant calling (germline/somatic)

### Create Your Own Pipeline

```bash
# Create a new pipeline using nf-core template
nf-core create
```

## Clean Up

In [ ]:
%%sh
# Remove work directory to free up space
# Only do this after you've verified results!
rm -rf ~/demo_test_run/work

echo "✅ Work directory cleaned up"
echo "Results and reports are preserved in ~/demo_test_run/"

## Visualize Results

View HTML reports directly in the notebook.

In [25]:
%%sh
# Find all HTML reports generated by the pipeline
echo "=== Available HTML Reports ==="
find ~/demo_test_run/results -name "*.html" -type f 2>/dev/null | while read file; do
    echo "$(basename $file): $file"
done

echo ""
find ~/demo_test_run -maxdepth 1 -name "*.html" -type f 2>/dev/null | while read file; do
    echo "$(basename $file): $file"
done

=== Available HTML Reports ===
pipeline_dag_2026-06-29_17-21-38.html: /home/sagemaker-user/demo_test_run/results/pipeline_info/pipeline_dag_2026-06-29_17-21-38.html
pipeline_dag_2026-06-29_17-25-09.html: /home/sagemaker-user/demo_test_run/results/pipeline_info/pipeline_dag_2026-06-29_17-25-09.html
pipeline_dag_2026-06-29_17-26-34.html: /home/sagemaker-user/demo_test_run/results/pipeline_info/pipeline_dag_2026-06-29_17-26-34.html
pipeline_dag_2026-06-29_17-43-30.html: /home/sagemaker-user/demo_test_run/results/pipeline_info/pipeline_dag_2026-06-29_17-43-30.html
execution_report_2026-06-29_17-43-30.html: /home/sagemaker-user/demo_test_run/results/pipeline_info/execution_report_2026-06-29_17-43-30.html
execution_timeline_2026-06-29_17-43-30.html: /home/sagemaker-user/demo_test_run/results/pipeline_info/execution_timeline_2026-06-29_17-43-30.html
SAMPLE2_PE_1_fastqc.html: /home/sagemaker-user/demo_test_run/results/fastqc/SAMPLE2_PE/SAMPLE2_PE_1_fastqc.html
SAMPLE2_PE_2_fastqc.html: /home/s